# Phase 3 – Bike Demand Prediction Model
**Sigmoid case study – Bike Sharing Demand**

**Goal:** predict rentals per hour (`cnt`) from calendar and weather information, evaluated the way it would be used – trained on the past, tested on the future – and then translate the forecast into an indicative number of bikes.

Runtime in Colab: about 6–8 minutes (gradient boosting is fitted several times for cross-validation). `Runtime → Run all`, then upload the ZIP when asked.

## 0. Setup

In [ ]:
import os, glob, zipfile, sys
if not (os.path.exists("data/hour.csv") and os.path.exists("data/day.csv")):
    try:
        from google.colab import files
        uploaded = files.upload()
        zip_name = [f for f in uploaded if f.lower().endswith(".zip")][0]
    except ImportError:
        zip_name = glob.glob("*.zip")[0]
    os.makedirs("data", exist_ok=True)
    with zipfile.ZipFile(zip_name) as z:
        for member in z.namelist():
            name = os.path.basename(member)
            if name in ("hour.csv", "day.csv", "Readme.txt"):
                with z.open(member) as src, open(os.path.join("data", name), "wb") as dst:
                    dst.write(src.read())
for d in ["src", "models", "outputs/charts", "outputs/model_results"]:
    os.makedirs(d, exist_ok=True)
print(sorted(os.listdir("data")))

In [ ]:
%%writefile src/preprocessing.py
"""
Loading and cleaning for the Bike Sharing dataset.

Rules followed:
* Raw CSVs are never modified on disk; every function returns a new DataFrame.
* Nothing is deleted. Bad values are corrected and FLAGGED so the change is visible.
* Every cleaning decision is listed in CLEANING_LOG (shown in the notebook / report).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

# Readme: values were divided by these maxima -> multiply back for human-readable units
TEMP_MAX_C, ATEMP_MAX_C, HUM_MAX, WIND_MAX = 41, 50, 100, 67

# Days on which the system was (partly) shut down or data are incomplete (< 22 recorded hours).
# Identified in Phase 1; kept for modelling but flagged.
DISRUPTED_DAYS = pd.to_datetime([
    "2011-01-18", "2011-01-26", "2011-01-27", "2011-02-22",
    "2011-08-27", "2011-08-28", "2012-10-29", "2012-10-30",
])

CLEANING_LOG = [
    ("hum == 0 on 2011-03-10 (22 hourly rows, 1 daily row)",
     "Physically impossible -> sensor error. Replaced with the mean humidity of the same hour on "
     "the previous and next day (daily file: mean of neighbouring days). Flag column `hum_imputed`."),
    ("weathersit == 4 (3 hourly rows)",
     "Too rare to learn from. New column `weather` merges 4 into 3 ('rain/snow'). Raw column kept."),
    ("windspeed == 0 (12.5% of hours; next value is 0.0896)",
     "Likely 'calm / not recorded'. Kept unchanged, flagged in `wind_zero`. Low impact on demand."),
    ("165 missing hours (mostly 02:00-05:00; storm shutdowns)",
     "Not filled with invented rows for EDA/model training. Zero-rental hours appear to be absent, "
     "so night-time averages are slightly overstated. Storm days flagged in `disrupted_day`."),
    ("High cnt values (up to 977/hr)",
     "Genuine commute peaks (Phase 1, Step 10). Not removed."),
]


def load_hour(path: str = "data/hour.csv") -> pd.DataFrame:
    return pd.read_csv(path, parse_dates=["dteday"])


def load_day(path: str = "data/day.csv") -> pd.DataFrame:
    return pd.read_csv(path, parse_dates=["dteday"])


def _impute_hourly_humidity(df: pd.DataFrame) -> pd.DataFrame:
    bad = df["hum"] == 0
    df["hum_imputed"] = bad.astype(int)
    if not bad.any():
        return df
    lookup = df.loc[~bad].set_index(["dteday", "hr"])["hum"]
    for idx in df.index[bad]:
        d, h = df.at[idx, "dteday"], df.at[idx, "hr"]
        neighbours = [lookup.get((d + pd.Timedelta(days=k), h)) for k in (-1, 1)]
        neighbours = [v for v in neighbours if v is not None and not pd.isna(v)]
        df.at[idx, "hum"] = np.mean(neighbours) if neighbours else df.loc[~bad, "hum"].median()
    return df


def clean_hour(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy().sort_values(["dteday", "hr"]).reset_index(drop=True)
    df["timestamp"] = df["dteday"] + pd.to_timedelta(df["hr"], unit="h")
    df = _impute_hourly_humidity(df)
    df["weather"] = df["weathersit"].clip(upper=3)
    df["wind_zero"] = (df["windspeed"] == 0).astype(int)
    df["disrupted_day"] = df["dteday"].isin(DISRUPTED_DAYS).astype(int)
    return add_physical_units(df)


def clean_day(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy().sort_values("dteday").reset_index(drop=True)
    bad = df["hum"] == 0
    df["hum_imputed"] = bad.astype(int)
    for idx in df.index[bad]:
        neigh = df.loc[[i for i in (idx - 1, idx + 1) if i in df.index and not bad[i]], "hum"]
        df.at[idx, "hum"] = neigh.mean()
    df["weather"] = df["weathersit"].clip(upper=3)
    df["disrupted_day"] = df["dteday"].isin(DISRUPTED_DAYS).astype(int)
    return add_physical_units(df)


def add_physical_units(df: pd.DataFrame) -> pd.DataFrame:
    """De-normalise for readable charts and for the prototype (no new information)."""
    df["temp_c"] = df["temp"] * TEMP_MAX_C
    df["atemp_c"] = df["atemp"] * ATEMP_MAX_C
    df["hum_pct"] = df["hum"] * HUM_MAX
    df["wind_unscaled"] = df["windspeed"] * WIND_MAX   # Readme does not state the unit
    return df


In [ ]:
%%writefile src/feature_engineering.py
"""
Feature engineering. Every feature uses only information known BEFORE the hour starts
(calendar + weather for that hour, which in deployment comes from a weather forecast).
No feature uses `casual`, `registered`, or future rows.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

MORNING_PEAK = (7, 8, 9)
EVENING_PEAK = (16, 17, 18, 19)
PEAK_HOURS = MORNING_PEAK + EVENING_PEAK
NIGHT_HOURS = (0, 1, 2, 3, 4, 5)

FEATURE_RATIONALE = {
    "hr":            "Hour of day - the single strongest driver (commute vs night).",
    "weekday":       "Day of week (0=Sun). Captures Fri/Mon effects beyond the working-day flag.",
    "workingday":    "Commuting happens on working days; changes the whole daily shape.",
    "holiday":       "Public holidays behave like weekends even on Mon-Fri.",
    "is_weekend":    "Sat/Sun indicator - leisure pattern (midday peak).",
    "mnth":          "Month - seasonal temperature/daylight cycle.",
    "season":        "Coarser seasonal signal from the source data.",
    "yr":            "Growth of the system (2012 ~ +63%). Lets the model learn the level shift.",
    "morning_peak":  "Working day AND 07-09 - morning commute window.",
    "evening_peak":  "Working day AND 16-19 - evening commute window.",
    "rush_hour":     "Either commute window on a working day - where shortages are most likely.",
    "night":         "00-05 - near-zero demand; useful window for rebalancing trucks.",
    "hr_sin/hr_cos": "Cyclical hour: tells linear models that 23:00 is next to 00:00.",
    "mnth_sin/mnth_cos": "Cyclical month: December is next to January.",
    "weather":       "1 clear, 2 mist/cloud, 3 rain/snow (4 merged into 3 - only 3 rows).",
    "temp":          "Temperature (normalised). Riding is weather-sensitive.",
    "temp_sq":       "Lets a linear model capture 'too hot' - demand flattens/falls at high temps.",
    "hum":           "Humidity - proxy for discomfort and rain.",
    "windspeed":     "Wind - discomfort on a bike.",
}

EXCLUDED_COLUMNS = {
    "casual":      "Leakage: casual + registered = cnt exactly. Unknown before the hour ends.",
    "registered":  "Leakage: same reason as casual.",
    "cnt":         "Target.",
    "instant":     "Row index; it is a disguised time trend with no causal meaning. `yr` covers the trend explicitly.",
    "dteday":      "Raw date string; decomposed into yr/mnth/weekday instead.",
    "timestamp":   "Helper for sorting/splitting, not a feature.",
    "atemp":       "'Feels-like' temperature, correlation with temp ~0.99 -> redundant; kept out to avoid collinearity.",
    "weathersit":  "Replaced by `weather` (category 4 merged into 3).",
    "hum_imputed / wind_zero / disrupted_day": "Data-quality flags for analysis, not demand drivers.",
    "temp_c / atemp_c / hum_pct / wind_unscaled": "Same information as the scaled columns, used only for readable charts/app.",
}

MODEL_FEATURES = [
    "hr", "weekday", "workingday", "holiday", "is_weekend", "mnth", "season", "yr",
    "morning_peak", "evening_peak", "rush_hour", "night",
    "hr_sin", "hr_cos", "mnth_sin", "mnth_cos",
    "weather", "temp", "temp_sq", "hum", "windspeed",
]


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["is_weekend"] = out["weekday"].isin([0, 6]).astype(int)
    if "hr" in out:
        hr = out["hr"]
        out["morning_peak"] = (hr.isin(MORNING_PEAK) & (out["workingday"] == 1)).astype(int)
        out["evening_peak"] = (hr.isin(EVENING_PEAK) & (out["workingday"] == 1)).astype(int)
        out["rush_hour"] = (out["morning_peak"] | out["evening_peak"]).astype(int)
        out["night"] = hr.isin(NIGHT_HOURS).astype(int)
        out["hr_sin"] = np.sin(2 * np.pi * hr / 24)
        out["hr_cos"] = np.cos(2 * np.pi * hr / 24)
    out["mnth_sin"] = np.sin(2 * np.pi * (out["mnth"] - 1) / 12)
    out["mnth_cos"] = np.cos(2 * np.pi * (out["mnth"] - 1) / 12)
    out["temp_sq"] = out["temp"] ** 2
    out["day_type"] = np.where(out["workingday"] == 1, "Working day", "Non-working day")
    return out


def daily_peak_share(hour_df: pd.DataFrame, min_hours: int = 22) -> pd.DataFrame:
    """Per day: share of rentals in the commute windows (07-09, 16-19), for every day type.

    Uses clock-hour windows (not the working-day-only flags) so working and non-working days
    are measured on the same 7 hours. Days with < `min_hours` recorded hours are dropped,
    because a partial day distorts the share.
    """
    g = hour_df.assign(in_peak=hour_df["hr"].isin(PEAK_HOURS) * hour_df["cnt"])
    daily = g.groupby("dteday").agg(
        cnt=("cnt", "sum"), peak_cnt=("in_peak", "sum"), n_hours=("hr", "size"),
        workingday=("workingday", "first"), yr=("yr", "first"), season=("season", "first"),
        mnth=("mnth", "first"), weather_mean=("weather", "mean"), temp=("temp", "mean"),
        hum=("hum", "mean"), windspeed=("windspeed", "mean"),
    ).reset_index()
    daily = daily[daily["n_hours"] >= min_hours].copy()
    daily["peak_share"] = daily["peak_cnt"] / daily["cnt"]
    return daily


In [ ]:
%%writefile src/model_training.py
"""
Model training and evaluation for hourly bike demand (target = cnt).

Design rules
* Chronological evaluation only: train <= 2012-06-30, test = 2012-07-01 .. 2012-12-31.
* Model selection uses expanding-window CV inside the training period (monthly folds Mar-Jun 2012).
  The test set is used once, for the chosen model.
* Inputs = feature_engineering.MODEL_FEATURES (no casual/registered/instant).
"""
from __future__ import annotations

import time

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

import feature_engineering as fe

SEED = 42
TRAIN_END = pd.Timestamp("2012-06-30")
TEST_START = pd.Timestamp("2012-07-01")
CV_MONTHS = ["2012-03", "2012-04", "2012-05", "2012-06"]   # expanding-window validation months
LAG_FEATURES = ["lag_168", "same_hour_4wk_mean"]


# ------------------------------------------------------------------ splitting
def chrono_split(df: pd.DataFrame):
    train = df[df["dteday"] <= TRAIN_END].copy()
    test = df[df["dteday"] >= TEST_START].copy()
    return train, test


def time_folds(train: pd.DataFrame):
    """Yield (fold_name, train_idx, valid_idx): fit on everything before the month, validate on the month."""
    for m in CV_MONTHS:
        start = pd.Timestamp(m + "-01")
        end = start + pd.offsets.MonthEnd(0)
        tr = train.index[train["dteday"] < start]
        va = train.index[(train["dteday"] >= start) & (train["dteday"] <= end)]
        yield m, tr, va


# ------------------------------------------------------------------ baselines
class HourProfileBaseline(BaseEstimator, RegressorMixin):
    """'What an ops manager would do without a model': average demand for this hour and day type,
    from the training data only."""

    def fit(self, X, y):
        d = pd.DataFrame({"hr": X["hr"].values, "wd": X["workingday"].values, "y": np.asarray(y)})
        self.table_ = d.groupby(["hr", "wd"]).y.mean()
        self.global_ = float(np.mean(y))
        return self

    def predict(self, X):
        keys = list(zip(X["hr"].values, X["workingday"].values))
        return np.array([self.table_.get(k, self.global_) for k in keys])


# ------------------------------------------------------------------ linear model features
def add_hour_daytype(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    X["hr_wd"] = X["hr"].astype(int) * 10 + X["workingday"].astype(int)   # 48 hour x day-type cells
    return X


LINEAR_CATEGORICAL = ["hr_wd", "weather", "mnth"]
LINEAR_NUMERIC = ["yr", "holiday", "temp", "temp_sq", "hum", "windspeed"]
# not used by the linear model: is_weekend (collinear with workingday+holiday), season (redundant with mnth),
# hr/weekday as numbers, peak flags and sin/cos (all absorbed by the 48 hour x day-type dummies)


def linear_pipeline() -> Pipeline:
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), LINEAR_CATEGORICAL),
        ("num", StandardScaler(), LINEAR_NUMERIC),
    ])
    return Pipeline([("hour_x_daytype", FunctionTransformer(add_hour_daytype)), ("prep", pre),
                     ("model", LinearRegression())])


def log_target(model):
    return TransformedTargetRegressor(regressor=model, func=np.log1p, inverse_func=np.expm1)


def candidate_models() -> dict:
    """Name -> unfitted estimator. Kept deliberately small: one or two settings per family."""
    rf = lambda leaf: RandomForestRegressor(n_estimators=300, min_samples_leaf=leaf, max_features=0.5,
                                            n_jobs=-1, random_state=SEED)
    gb = lambda depth: GradientBoostingRegressor(n_estimators=400, learning_rate=0.05, max_depth=depth,
                                                 subsample=0.8, random_state=SEED)
    return {
        "Dummy (training mean)": DummyRegressor(strategy="mean"),
        "Baseline: hour x day-type average": HourProfileBaseline(),
        "Linear regression": linear_pipeline(),
        "Linear regression (log target)": log_target(linear_pipeline()),
        "Decision tree (depth 10)": DecisionTreeRegressor(max_depth=10, min_samples_leaf=5, random_state=SEED),
        "Decision tree (depth 14)": DecisionTreeRegressor(max_depth=14, min_samples_leaf=5, random_state=SEED),
        "Random forest (leaf 1)": rf(1),
        "Random forest (leaf 3)": rf(3),
        "Random forest (leaf 3, log target)": log_target(rf(3)),
        "Gradient boosting (depth 4)": gb(4),
        "Gradient boosting (depth 6)": gb(6),
        "Gradient boosting (depth 6, log target)": log_target(gb(6)),
    }


# ------------------------------------------------------------------ metrics
def evaluate(y_true, y_pred, is_peak=None) -> dict:
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    err = y_pred - y_true
    out = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        # WAPE instead of MAPE: MAPE explodes on hours with 1-5 rentals
        "WAPE_%": 100 * np.abs(err).sum() / y_true.sum(),
        "Bias (mean pred - actual)": err.mean(),
    }
    if is_peak is not None:
        p = np.asarray(is_peak, bool)
        out["Peak MAE"] = np.abs(err[p]).mean()
        out["Peak bias"] = err[p].mean()
        out["Peak under-predicted %"] = 100 * (err[p] < 0).mean()
    return out


def cross_validate(models: dict, train: pd.DataFrame, features: list, target: str = "cnt") -> pd.DataFrame:
    rows = []
    for name, est in models.items():
        t0 = time.time()
        for fold, tr, va in time_folds(train):
            m = clone(est).fit(train.loc[tr, features], train.loc[tr, target])
            pred = np.clip(m.predict(train.loc[va, features]), 0, None)
            rows.append({"model": name, "fold": fold, **evaluate(train.loc[va, target], pred, train.loc[va, "rush_hour"])})
        rows[-1]["fit_seconds"] = time.time() - t0
    res = pd.DataFrame(rows)
    summary = res.groupby("model", sort=False).mean(numeric_only=True)
    return summary.sort_values("MAE"), res


# ------------------------------------------------------------------ lag features (variant)
def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    """Same hour last week, and mean of the same hour over the previous 4 weeks.

    Both are known 7 days in advance, so they are valid for forecasts up to a week ahead.
    Missing hours are treated as 0 rentals (Phase 1: absent hours are almost all 02:00-05:00 / storm closures).
    """
    s = df.set_index("timestamp")["cnt"]
    full = pd.date_range(s.index.min(), s.index.max(), freq="h")
    s = s.reindex(full).fillna(0)
    lags = pd.concat({k: s.shift(168 * k) for k in (1, 2, 3, 4)}, axis=1)
    feats = pd.DataFrame({"lag_168": lags[1], "same_hour_4wk_mean": lags.mean(axis=1, skipna=False)})
    out = df.join(feats, on="timestamp")
    return out


# ------------------------------------------------------------------ prediction intervals
def quantile_models(alpha_low=0.1, alpha_high=0.9) -> dict:
    q = lambda a: GradientBoostingRegressor(loss="quantile", alpha=a, n_estimators=400, learning_rate=0.05,
                                            max_depth=6, subsample=0.8, random_state=SEED)
    return {f"P{int(alpha_low * 100)}": q(alpha_low), f"P{int(alpha_high * 100)}": q(alpha_high)}


def out_of_fold_predictions(est, train: pd.DataFrame, features: list, target: str = "cnt") -> pd.Series:
    preds = []
    for _, tr, va in time_folds(train):
        m = clone(est).fit(train.loc[tr, features], train.loc[tr, target])
        preds.append(pd.Series(np.clip(m.predict(train.loc[va, features]), 0, None), index=va))
    return pd.concat(preds)


def planning_multipliers(actual: pd.Series, predicted: pd.Series, is_rush: pd.Series, q: float = 0.9) -> dict:
    """Empirical q-quantile of (actual+1)/(predicted+1) on validation folds, separately for rush / other hours.
    Planning forecast = (point + 1) * multiplier - 1  ->  covers ~q of hours if errors stay similar."""
    ratio = (np.asarray(actual) + 1) / (np.asarray(predicted) + 1)
    grp = np.where(np.asarray(is_rush) == 1, "rush", "other")
    return pd.Series(ratio, index=grp).groupby(level=0).quantile(q).to_dict()


def planning_forecast(point: np.ndarray, is_rush, multipliers: dict) -> np.ndarray:
    f = np.where(np.asarray(is_rush) == 1, multipliers["rush"], multipliers["other"])
    return np.clip((np.asarray(point) + 1) * f - 1, 0, None)


def walk_forward(est, df: pd.DataFrame, features: list, months: list, target: str = "cnt") -> pd.Series:
    """Retrain on all data before each month, predict that month (simulates monthly retraining)."""
    preds = []
    for m in months:
        start = pd.Timestamp(m + "-01")
        end = start + pd.offsets.MonthEnd(0)
        a = df[df["dteday"] < start]
        b = df[(df["dteday"] >= start) & (df["dteday"] <= end)]
        mm = clone(est).fit(a[features], a[target])
        preds.append(pd.Series(np.clip(mm.predict(b[features]), 0, None), index=b.index))
    return pd.concat(preds)


def bikes_in_use(rentals_per_hour, trip_minutes: float):
    """Little's law: average number of bikes on the road = arrival rate x average trip time."""
    return rentals_per_hour * trip_minutes / 60          # works for numbers, arrays, Series and DataFrames


def save_bundle(path: str, **items):
    import joblib, sklearn
    items["sklearn_version"] = sklearn.__version__
    joblib.dump(items, path)


def load_bundle(path: str) -> dict:
    import joblib
    return joblib.load(path)


# ------------------------------------------------------------------ scenarios (used by the prototype)
def season_for(date) -> int:
    """Season coding used in the dataset (verified on day.csv): changes on 21 Mar, 21 Jun, 23 Sep, 21 Dec."""
    md = (pd.Timestamp(date).month, pd.Timestamp(date).day)
    if md >= (12, 21) or md < (3, 21):
        return 1
    if md < (6, 21):
        return 2
    if md < (9, 23):
        return 3
    return 4


def scenario_frame(date, hours=range(24), weather: int = 1, temp_c: float | None = None,
                   hum_pct: float | None = None, wind: float | None = None, holiday: int = 0,
                   climate: pd.DataFrame | None = None, yr: int = 1) -> pd.DataFrame:
    """Build model-ready rows for a date/hours under a weather scenario.

    Missing weather values default to the historical median for that month and hour (`climate`).
    `yr` is capped at 1 (2012 demand level): the model has never seen a later year.
    """
    date = pd.Timestamp(date)
    rows = []
    for h in hours:
        clim = climate.loc[(date.month, h)] if climate is not None else None
        t = temp_c if temp_c is not None else clim["temp_c"]
        hu = hum_pct if hum_pct is not None else clim["hum_pct"]
        w = wind if wind is not None else clim["wind_unscaled"]
        weekday = (date.dayofweek + 1) % 7                     # pandas Mon=0 -> dataset Sun=0
        workingday = int(weekday in (1, 2, 3, 4, 5) and not holiday)
        rows.append({"dteday": date.normalize(), "hr": h, "weekday": weekday, "workingday": workingday,
                     "holiday": int(holiday), "mnth": date.month, "season": season_for(date), "yr": min(int(yr), 1),
                     "weather": int(weather), "temp": t / 41, "hum": hu / 100, "windspeed": w / 67})
    return fe.add_features(pd.DataFrame(rows))


def climate_table(hour_df: pd.DataFrame) -> pd.DataFrame:
    """Median temperature / humidity / wind by (month, hour) - default weather for scenarios."""
    return hour_df.groupby(["mnth", "hr"])[["temp_c", "hum_pct", "wind_unscaled"]].median()


In [ ]:
import importlib, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

sys.path.insert(0, "src")
import preprocessing as pp, feature_engineering as fe, model_training as mt
for m in (pp, fe, mt):
    importlib.reload(m)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
np.random.seed(mt.SEED)

BLUE, ORANGE, AQUA, RED = "#2a78d6", "#eb6834", "#1baf7a", "#e34948"
INK2, GRID, SURFACE = "#52514e", "#e6e5e0", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#b9b8b0", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "lines.linewidth": 2, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "legend.frameon": False,
})
def save(fig, name):
    fig.savefig(f"outputs/charts/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

## 1. Target, inputs and leakage check
- **Target:** `cnt` – rentals in the hour (casual + registered).
- **Inputs:** the 21 features from Phase 2 (`fe.MODEL_FEATURES`): calendar, peak flags, cyclical encodings, weather.
- **Excluded:** `casual`, `registered` (they add up to the target), `instant`, `atemp` (duplicate of `temp`), raw date. Lag features are tested separately in §4.

In [ ]:
hour = fe.add_features(pp.clean_hour(pp.load_hour("data/hour.csv")))
F = fe.MODEL_FEATURES
assert not set(F) & {"casual", "registered", "cnt", "instant", "atemp"}, "leakage!"

train, test = mt.chrono_split(hour)
print(f"Train: {train.dteday.min().date()} -> {train.dteday.max().date()}  ({len(train):,} hours)")
print(f"Test : {test.dteday.min().date()} -> {test.dteday.max().date()}  ({len(test):,} hours)  <- used ONCE, for the chosen model")
print("Validation folds (expanding window inside train):", mt.CV_MONTHS)
print(f"\nAverage cnt  train: {train.cnt.mean():.1f} | test: {test.cnt.mean():.1f}")
print(f"Jul-Dec 2011 avg: {hour[(hour.yr == 0) & (hour.mnth >= 7)].cnt.mean():.1f} | Jan-Jun 2012 avg: {train[train.yr == 1].cnt.mean():.1f}")

**Why this split.** Training covers Jan 2011–Jun 2012 (13,003 hours); the test set is Jul–Dec 2012 (4,376 hours), untouched until the final model is chosen. Training therefore includes six months of 2012, so the model sees the growth (`yr = 1`) before it is tested.

The test period is hard: its average demand (**254/hr**) is **52% above** the training average (168/hr), because it combines the busiest season with continued growth. Any model trained on the past will be tempted to under-predict it – a realistic stress test.

**Model selection** uses four expanding-window folds (validate on Mar, Apr, May, Jun 2012, each time training only on earlier data). The test set plays no part in choosing the model.

## 2. Baselines and candidate models – time-ordered cross-validation

In [ ]:
t0 = time.time()
cv_summary, cv_folds = mt.cross_validate(mt.candidate_models(), train, F)
print(f"CV finished in {time.time() - t0:.0f}s")
cv_summary.to_csv("outputs/model_results/cv_comparison.csv")
display(cv_summary.drop(columns="fit_seconds").round(2))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
order = cv_summary.index[::-1]
ax[0].barh(order, cv_summary.loc[order, "MAE"], color=BLUE, edgecolor=SURFACE, linewidth=1.5)
ax[0].set(title="Validation MAE (all hours)", xlabel="rentals per hour")
ax[1].barh(order, cv_summary.loc[order, "Peak MAE"], color=ORANGE, edgecolor=SURFACE, linewidth=1.5)
ax[1].set(title="Validation MAE (working-day rush hours)", xlabel="rentals per hour")
for a, col in zip(ax, ["MAE", "Peak MAE"]):
    for y, v in enumerate(cv_summary.loc[order, col]):
        a.text(v + 2, y, f"{v:.0f}", va="center", fontsize=9, color=INK2)
plt.tight_layout(); save(fig, "15_model_comparison_cv")

**Validation results (average over the four months):**
- **Gradient boosting** is clearly best: MAE **43** rentals/hr, R² 0.89.
- Random forest ≈ 51, decision tree ≈ 56, linear regression 66 (log target) to 81.
- **Business baseline** ("average for this hour and day type") ≈ **110**; predicting the overall mean ≈ 178.
- Gradient boosting cuts the error of the business baseline by about **60%**.

**What the metrics mean:**
- **MAE:** the average size of the miss, in rentals per hour.
- **RMSE:** like MAE but penalises big misses more, which is relevant because big misses at peaks are the costly ones.
- **R²:** the share of hour-to-hour variation explained.
- **WAPE:** total absolute error ÷ total rentals. I use it instead of MAPE, because MAPE explodes on night hours with 1–5 rentals.
- **Bias:** average (prediction − actual); negative means under-prediction.

**Pattern to notice:** nearly every model has a **negative bias**, strongest at peaks. Demand kept rising through spring 2012, faster than the earlier training data implied. Tree models cannot extrapolate beyond the range they have seen; this matters for the test set too.

## 3. Model choice

**Chosen: Gradient boosting, depth 6, raw target (400 trees, learning rate 0.05, subsample 0.8, seed 42).**

**Why not the log-target version, which has the lowest MAE (42.4 vs 43.3)?**
- Its under-prediction is larger (bias −16 vs −9).
- It is worse at rush hours (peak MAE 77 vs 74) and on RMSE (66.5 vs 66.0).
- A log-target model predicts something closer to the median than the mean, so it systematically shades forecasts down.

For this business, under-predicting a peak means empty stations, which is the costly error. A 0.9-rental gain in average MAE is not worth more bias at the peaks.

**Why not tune more?** The step from depth 4 to depth 6 gained 4 MAE points. Further tuning would gain far less than the structural issue (growth), and would risk over-fitting the four validation months. Phase 4/5 time is better spent elsewhere.

**Why gradient boosting beats the others:**
- The demand shape depends on **interactions**: hour × day type × weather × season. Boosting builds these from many small trees.
- A single tree is too coarse.
- The random forest averages deep trees and shrinks peaks the most (peak bias −58).
- The linear model needs every interaction spelled out by hand.

## 4. Do lag features help? ("same hour last week")

In [ ]:
hour_lag = mt.add_lag_features(hour)
train_lag, test_lag = mt.chrono_split(hour_lag)
train_lag = train_lag.dropna(subset=mt.LAG_FEATURES)        # first 4 weeks of 2011 have no history
best = mt.candidate_models()["Gradient boosting (depth 6)"]
s_no, _ = mt.cross_validate({"GB depth 6 – no lags": best}, train_lag, F)
s_lag, _ = mt.cross_validate({"GB depth 6 – with lags": best}, train_lag, F + mt.LAG_FEATURES)
lag_cmp = pd.concat([s_no, s_lag]).drop(columns="fit_seconds")
display(lag_cmp.round(2)); lag_cmp.to_csv("outputs/model_results/lag_variant_cv.csv")

**Result.** Adding "same hour last week" and the 4-week average for the same hour changes MAE by only **−0.3** (43.6 → 43.3). It makes RMSE (66.4 → 67.5), peak MAE (71.9 → 75.2) and peak bias (−20 → −32) **worse**.

**Decision: no lag features.** They add little because calendar + weather already explain the repeating weekly pattern. They would also limit the prototype: a lag model can't answer "what if next Friday is rainy?" without last week's actuals, and it can only forecast 7 days ahead.

> **If asked:** "I tested lags rather than assuming they help. They didn't improve validation error and they reduce what the model can be used for, so I left them out."

## 5. Final evaluation on the untouched test set (Jul–Dec 2012)

In [ ]:
final_name = "Gradient boosting (depth 6)"
models = mt.candidate_models()
show = ["Dummy (training mean)", "Baseline: hour x day-type average", "Linear regression (log target)",
        "Decision tree (depth 14)", "Random forest (leaf 1)", final_name]
test_rows, fitted = [], {}
for name in show:
    m = clone(models[name]).fit(train[F], train.cnt)
    fitted[name] = m
    p = np.clip(m.predict(test[F]), 0, None)
    test_rows.append({"model": name, **mt.evaluate(test.cnt, p, test.rush_hour)})
test_table = pd.DataFrame(test_rows).set_index("model")
test_table.to_csv("outputs/model_results/test_metrics.csv")
display(test_table.round(2))

final_model = fitted[final_name]
test = test.assign(pred=np.clip(final_model.predict(test[F]), 0, None))
test["resid"] = test.pred - test.cnt

**Test-set results (Jul–Dec 2012, used once):**

| Metric | Gradient boosting | Business baseline |
|---|---|---|
| MAE | **43.1** rentals/hr | 107.2 |
| RMSE | 66.1 | 150.4 |
| R² | **0.91** | 0.53 |
| WAPE | **17.0%** | 42.2% |
| Bias | **−21** (under-predicts) | −87 |
| Rush-hour MAE | 80.8 (≈ 15.5% of the average rush-hour volume of 523) | 218.6 |
| Rush hours under-predicted | **81%** | 91% |

Test MAE (43.1) matches validation (43.3), so the model is not over-fitted to the validation months. The other models rank the same way on the test set (shown for information only; they were not used to choose).

**Is the error acceptable?** For planning a day's rebalancing, a ~15–17% error is usable. But the **systematic under-prediction at peaks** means the point forecast should **not** be used directly as a stock target. §7 adds a planning buffer.

### 5.1 Actual vs predicted

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"width_ratios": [1, 2]})
ax[0].scatter(test.cnt, test.pred, s=6, alpha=.25, color=BLUE, linewidths=0)
lim = max(test.cnt.max(), test.pred.max()) * 1.02
ax[0].plot([0, lim], [0, lim], color=INK2, lw=1, ls="--"); ax[0].set(xlim=(0, lim), ylim=(0, lim))
ax[0].set(title="Test hours: predicted vs actual", xlabel="actual rentals", ylabel="predicted rentals")
wk = test[(test.dteday >= "2012-09-10") & (test.dteday <= "2012-09-23")]
ax[1].plot(wk.timestamp, wk.cnt, color=INK2, lw=1.4, label="actual")
ax[1].plot(wk.timestamp, wk.pred, color=BLUE, lw=1.8, label="predicted")
ax[1].set(title="Two test weeks (10–23 Sep 2012)", ylabel="rentals per hour"); ax[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2)
plt.tight_layout(); save(fig, "16_actual_vs_predicted")

### 5.2 Residual analysis – where and when is the model wrong?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(test.pred, test.resid, s=6, alpha=.25, color=BLUE, linewidths=0); ax[0].axhline(0, color=INK2, lw=1)
ax[0].set(title="Residual vs prediction", xlabel="predicted rentals", ylabel="predicted − actual")
byhr = test.groupby(["day_type", "hr"]).resid.mean().unstack(0)
ax[1].plot(byhr.index, byhr["Working day"], color=BLUE, marker="o", markersize=3, label="Working day")
ax[1].plot(byhr.index, byhr["Non-working day"], color=ORANGE, marker="o", markersize=3, label="Non-working day")
ax[1].axhline(0, color=INK2, lw=1); ax[1].set(title="Mean residual by hour (− = under-predict)", xlabel="hour", xticks=range(0, 24, 2)); ax[1].legend()
bym = test.groupby("mnth").resid.mean()
ax[2].bar(bym.index, bym.values, color=[RED if v < 0 else BLUE for v in bym.values], edgecolor=SURFACE, linewidth=1.5)
ax[2].axhline(0, color=INK2, lw=1); ax[2].set(title="Mean residual by test month", xlabel="month", ylabel="predicted − actual")
plt.tight_layout(); save(fig, "17_residuals")

print("Mean residual by month:", bym.round(1).to_dict())
print("Mean residual at working-day 08:00 / 17:00 / 18:00:", byhr.loc[[8, 17, 18], "Working day"].round(1).to_dict())
sandy = test[test.dteday.isin(pd.to_datetime(["2012-10-29", "2012-10-30"]))]
print(f"Hurricane Sandy days: actual {sandy.cnt.sum()}, predicted {sandy.pred.sum():.0f} (model has no storm information)")

**Where the model is wrong:**
- **Working-day peaks** are under-predicted: −68 at 08:00, −78 at 17:00, −62 at 18:00. Non-working days are close to zero bias.
- **By month:** worst in Sep–Oct (−28, −41), almost unbiased by December (−3).
- **Residuals fan out** as demand rises: errors are roughly proportional to volume. This is why the planning buffer in §7 is a multiplier, not a fixed number.
- **Hurricane Sandy** (29–30 Oct): actual 1,118 rentals vs 1,826 predicted. The model has no storm/closure information, so it over-predicts disruptions. In operations, a manual override for known events is needed.

**Why the under-prediction?** No training row has both `yr = 1` and a July–December month. The trees must combine "2012 level" (learned from Jan–Jun) with "autumn pattern" (learned from 2011), and tree ensembles do this only partially. Demand in autumn 2012 was higher than that combination suggests. The fix is operational: **retrain monthly** (§6 shows this halves the bias).

### 5.3 Error during peak demand

In [ ]:
test["hour_group"] = np.select(
    [test.rush_hour == 1, test.night == 1, test.workingday == 1],
    ["1 Working-day rush (07-09, 16-19)", "4 Night (00-05)", "2 Working-day other hours"],
    default="3 Non-working day (06-23)")
g = test.assign(abs_err=test.resid.abs(), under=100 * (test.resid < 0))
peak_tbl = g.groupby("hour_group").agg(hours=("cnt", "size"), avg_actual=("cnt", "mean"), MAE=("abs_err", "mean"),
                                       bias=("resid", "mean"), pct_hours_under_predicted=("under", "mean"))
peak_tbl.insert(3, "MAE_as_%_of_actual", 100 * peak_tbl.MAE / peak_tbl.avg_actual)
display(peak_tbl.round(1)); peak_tbl.to_csv("outputs/model_results/peak_error_analysis.csv")
top = test.nlargest(int(len(test) * .05), "cnt")
print(f"Busiest 5% of test hours (cnt >= {top.cnt.min()}): MAE {top.resid.abs().mean():.0f}, bias {top.resid.mean():.0f}, "
      f"under-predicted {100 * (top.resid < 0).mean():.0f}%")

**Peak-demand error analysis:**
- **Rush hours:** MAE 80.8, which is **15.5%** of the average actual. Relative to volume, that is the same accuracy as other working hours (15.6%), but the misses are mostly **under-predictions (81%)**.
- **Non-working days:** 19% relative error, with little bias.
- **Night:** 30% relative error, but on only ~34 rentals/hr, so it is operationally irrelevant.
- **The busiest 5% of hours** (≥ 704 rentals) are under-predicted by **124 rentals on average, in 98% of cases**. The model "regresses to the mean" on extreme hours, which are exactly the hours that cause complaints.

**So:** the model is good at *when* and *roughly how much*, and weak at the very top of the peaks. That is the strongest argument for planning with a P90 forecast rather than the point forecast.

### 5.4 What drives the predictions? (permutation importance on the test set)

In [ ]:
sample = test.sample(3000, random_state=mt.SEED)
pi = permutation_importance(final_model, sample[F], sample.cnt, n_repeats=5, random_state=mt.SEED,
                            scoring="neg_mean_absolute_error", n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=F).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp.index, imp.values, color=BLUE, edgecolor=SURFACE, linewidth=1.5)
ax.set(title="Permutation importance: MAE increase when a feature is shuffled", xlabel="extra MAE (rentals per hour)")
save(fig, "18_feature_importance")
print(imp.sort_values(ascending=False).round(1).head(10).to_string())
imp.sort_values(ascending=False).to_csv("outputs/model_results/permutation_importance.csv")

**What drives the predictions:**
- Time of day dominates: shuffling `hr` adds **78** to MAE; `rush_hour` adds 38; `hr_sin`/`hr_cos` add 24 and 23.
- Next come temperature (`temp_sq` 12, `temp` 7), `workingday` (11) and humidity (10).
- Weather category, season, month and wind matter much less once the others are known.

**Two caveats an interviewer may probe:**
1. **`yr` shows 0 importance** only because the test sample is all 2012. Shuffling a column that has a single value changes nothing. That doesn't mean growth is unimportant (§6 shows it is critical).
2. Related features share credit (`hr`, `hr_sin`, `hr_cos`, `rush_hour` all describe the hour), so individual scores understate the "time of day" family as a whole.

**Business reading:** *when* (hour × day type) explains most of demand; weather and season scale it. That is exactly what the operations plan needs: a daily shape, adjusted by the forecast and the season.

## 6. Why a chronological split? (evidence, not just a rule)

In [ ]:
rows = []
# (a) chosen design
rows.append({"evaluation design": "Chronological: train to Jun-2012, test Jul–Dec 2012 (USED)", **mt.evaluate(test.cnt, test.pred)})
# (b) random 75/25 split of all hours
Xa, Xb, ya, yb = train_test_split(hour[F], hour.cnt, test_size=.25, random_state=mt.SEED)
pr = np.clip(clone(models[final_name]).fit(Xa, ya).predict(Xb), 0, None)
rows.append({"evaluation design": "Random 75/25 split (NOT used)", **mt.evaluate(yb, pr)})
# (c) train 2011 only, test 2012
a, b = hour[hour.yr == 0], hour[hour.yr == 1]
p11 = np.clip(clone(models[final_name]).fit(a[F], a.cnt).predict(b[F]), 0, None)
rows.append({"evaluation design": "Train 2011 -> test 2012 (NOT used)", **mt.evaluate(b.cnt, p11)})
# (d) realistic deployment: retrain every month
wf = mt.walk_forward(models[final_name], hour, F, ["2012-07", "2012-08", "2012-09", "2012-10", "2012-11", "2012-12"]).reindex(test.index)
test["pred_wf"] = wf
rows.append({"evaluation design": "Walk-forward: retrain monthly, predict next month (deployment)", **mt.evaluate(test.cnt, wf)})
split_cmp = pd.DataFrame(rows).set_index("evaluation design")
display(split_cmp.round(2)); split_cmp.to_csv("outputs/model_results/split_comparison.csv")
print("Walk-forward peak metrics:", {k: round(v, 1) for k, v in mt.evaluate(test.cnt, wf, test.rush_hour).items() if "Peak" in k})

**Same model, different evaluation designs:**

| Design | MAE | R² | Bias |
|---|---|---|---|
| **Chronological (used)** | **43.1** | 0.91 | −21 |
| Random 75/25 split | 23.0 | 0.96 | +1 |
| Train 2011 → test 2012 | 89.3 | 0.64 | −85 |
| Walk-forward, monthly retrain | **39.0** | 0.92 | **−10** |

- **The random split looks twice as good, but it is misleading.** Hours from the same day (and the same growth level) end up in both train and test, so the model is effectively interpolating, not forecasting. Real operations always predict days that haven't happened yet.
- **2011 → 2012** fails because the model has never seen the 2012 level, which confirms the Phase 1 concern.
- **Walk-forward** retrains every month on all data to date, which is how the model would actually run. It cuts MAE to **39** and halves the bias (peak bias −22 vs −50). **Recommendation: retrain monthly.**

## 7. Planning for uncertainty – a P90 "planning forecast"

In [ ]:
# Attempt 1: quantile gradient boosting (P10 / P90)
qm = {k: clone(v).fit(train[F], train.cnt) for k, v in mt.quantile_models().items()}
test["q10"], test["q90"] = qm["P10"].predict(test[F]), qm["P90"].predict(test[F])
cov_q90 = (test.cnt <= test.q90).mean()

# Attempt 2: point forecast x multiplier calibrated on the validation folds (no test data used)
oof = mt.out_of_fold_predictions(models[final_name], train, F)
mult = mt.planning_multipliers(train.loc[oof.index, "cnt"], oof, train.loc[oof.index, "rush_hour"], q=0.9)
test["plan90"] = mt.planning_forecast(test.pred, test.rush_hour, mult)
cov_plan = (test.cnt <= test.plan90).mean()

cov = pd.DataFrame({
    "all hours": [cov_q90, cov_plan],
    "working-day rush hours": [(test.cnt <= test.q90)[test.rush_hour == 1].mean(), (test.cnt <= test.plan90)[test.rush_hour == 1].mean()],
    "avg uplift vs point forecast": [(test.q90 - test.pred).mean(), (test.plan90 - test.pred).mean()],
}, index=["Quantile GB (P90)", "Calibrated multiplier (P90)"])
print("Calibrated multipliers (from validation folds):", {k: round(v, 3) for k, v in mult.items()})
display(cov.round(3)); cov.to_csv("outputs/model_results/planning_forecast_coverage.csv")

fig, ax = plt.subplots(figsize=(12, 3.8))
wk = test[(test.dteday >= "2012-09-10") & (test.dteday <= "2012-09-16")]
ax.fill_between(wk.timestamp, wk.pred, wk.plan90, color=BLUE, alpha=.18, linewidth=0, label="buffer to P90 planning forecast")
ax.plot(wk.timestamp, wk.pred, color=BLUE, lw=1.8, label="point forecast")
ax.plot(wk.timestamp, wk.cnt, color=INK2, lw=1.3, label="actual")
ax.set(title="Planning forecast covers ~90% of hours (test week 10–16 Sep 2012)", ylabel="rentals per hour"); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3)
save(fig, "19_planning_forecast")

**Two attempts at a "safe" planning number:**
1. **Quantile gradient boosting (P90)** should cover 90% of hours, but covers only **65%** (47% at rush hours). It learned the training-period level and inherits the same growth problem. **Rejected.**
2. **Calibrated multiplier.** On the validation months (Mar–Jun 2012, no test data), actual demand was at most **1.37× the forecast in 90% of rush hours** (1.46× in other hours, where small counts are noisier). Applying these multipliers to the test forecasts covers **89.4%** of all hours and **95.6%** of rush hours, close to the 90% target. Average uplift: +99 rentals/hr.

**How to use it:**
- **Point forecast:** expected demand, for reporting.
- **P90 planning forecast:** how many bikes to have *available* at peak stations, so that in roughly 9 out of 10 peak hours demand is covered.

The price is extra bikes positioned off-peak – a deliberate trade, because running out is costlier than a few idle bikes.

## 8. From rentals to bikes – an indicative framework (with labelled assumptions)

**Three different quantities – don't mix them up:**

| Quantity | What it is | Do we have it? |
|---|---|---|
| Rental demand | rentals starting in an hour | **Yes** – the model predicts this |
| Bikes in use at the same time | bikes on the road at a given moment | **Estimated** – needs average trip time |
| Fleet size | bikes owned = in use + parked as station stock + in repair | **No** – needs station inventory & maintenance data |

**Little's law:** bikes on the road ≈ rentals per hour × average trip time (hours). Trip duration is **not in this dataset**, so it is an **assumption** (shown for 15 / 20 / 30 minutes – replace with the real average from trip logs).

In [ ]:
TRIP_MIN = [15, 20, 30]
wd = test[(test.workingday == 1) & (test.disrupted_day == 0)]
prof = wd.groupby("hr")[["cnt", "pred", "plan90"]].mean()
busiest = wd.groupby("dteday").plan90.max()

rows = []
for w in TRIP_MIN:
    rows.append({"avg trip (min, ASSUMED)": w,
                 "17:00 avg actual -> bikes on road": mt.bikes_in_use(prof.loc[17, "cnt"], w),
                 "17:00 P90 plan -> bikes on road": mt.bikes_in_use(prof.loc[17, "plan90"], w),
                 "busiest test day P90 -> bikes on road": mt.bikes_in_use(busiest.max(), w),
                 "13:00 avg actual -> bikes on road": mt.bikes_in_use(prof.loc[13, "cnt"], w),
                 "03:00 avg actual -> bikes on road": mt.bikes_in_use(prof.loc[3, "cnt"], w)})
little = pd.DataFrame(rows).set_index("avg trip (min, ASSUMED)")
display(little.round(0)); little.to_csv("outputs/model_results/bikes_in_use_littles_law.csv")

w = 20
inuse = mt.bikes_in_use(prof, w)
peak_need = inuse.plan90.max()
print(f"With a {w}-min trip assumption: P90 bikes on the road peaks at {peak_need:.0f} (hour {inuse.plan90.idxmax()}:00) on an average working day.")
print(f"Share of that peak requirement that is NOT on the road (avg actual) at 13:00: {1 - inuse.loc[13, 'cnt'] / peak_need:.0%}, at 03:00: {1 - inuse.loc[3, 'cnt'] / peak_need:.0%}")
print(f"Busiest test day: P90 peak {busiest.max():.0f} rentals/hr on {busiest.idxmax().date()} -> {mt.bikes_in_use(busiest.max(), w):.0f} bikes on the road")

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(inuse.index, inuse.cnt, inuse.plan90.max(), color=ORANGE, alpha=.15, linewidth=0,
                label="bikes parked (if fleet on road = P90 peak)")
ax.plot(inuse.index, inuse.plan90, color=BLUE, ls="--", lw=1.6, label="P90 planning forecast")
ax.plot(inuse.index, inuse.cnt, color=BLUE, marker="o", markersize=4, label="actual (avg working day, test)")
ax.axhline(inuse.plan90.max(), color=INK2, lw=1)
ax.set(title=f"Bikes on the road by hour – working days, Jul–Dec 2012 (assumes {w}-min trips)", xlabel="hour of day",
       ylabel="bikes on the road (indicative)", xticks=range(24)); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3)
save(fig, "20_bikes_on_road")

**Indicative translation (trip time is an assumption; replace it with the real average from trip logs):**
- With **20-minute trips**, an average working day in Jul–Dec 2012 had about **241 bikes on the road at 17:00**. The P90 planning level is **292**.
- On the **busiest test day** (11 Sep 2012), the P90 peak of 1,181 rentals/hr means about **394 bikes on the road**.
- **Sensitivity:** with 15-minute trips the 17:00 P90 is 219; with 30-minute trips it is 439. The trip-time assumption matters as much as the model.

**Why bikes sit idle – the key number:** if enough bikes are on the road for the 17:00 P90 peak, then **69% of them are parked at 13:00 and 99% at 03:00**. Idle bikes at midday are a built-in consequence of a two-peak demand pattern. They are **not** evidence of too many bikes, and buying more would make midday idleness worse.

**What we can't calculate:** fleet size = bikes on the road + bikes parked as station stock (so a rider finds one) + bikes in repair. Station stock depends on how many stations there are and how unevenly demand is spread across them, which needs station data. **Decision test:** compare the peak bikes-on-road figure with the company's actual fleet. If the fleet is several times larger, a peak shortage must be a placement problem, not a quantity problem.

## 9. Scenario predictions (used by the prototype)

In [ ]:
# Refit the chosen model on ALL data (2011-2012) for deployment. Evaluation above used the train-only model.
deploy_model = clone(models[final_name]).fit(hour[F], hour.cnt)
climate = mt.climate_table(hour)

scen = []
for month in range(1, 13):
    for label, date in [("Working day", pd.Timestamp(2012, month, 1) + pd.offsets.Week(weekday=2)),      # a Wednesday
                        ("Non-working day", pd.Timestamp(2012, month, 1) + pd.offsets.Week(weekday=5))]:  # a Saturday
        for wthr in (1, 2, 3):
            fr = mt.scenario_frame(date, weather=wthr, climate=climate)
            p = np.clip(deploy_model.predict(fr[F]), 0, None)
            scen.append(pd.DataFrame({"month": month, "day_type": label, "weather": wthr, "hr": fr.hr,
                                      "point": p, "plan90": mt.planning_forecast(p, fr.rush_hour, mult)}))
scen = pd.concat(scen, ignore_index=True)
scen.to_csv("outputs/model_results/scenario_predictions.csv", index=False)
pv = scen[(scen.weather == 1)].pivot_table(index="month", columns="day_type", values="point", aggfunc="max")
print("Predicted busiest hour (clear weather, typical temperature for the month, 2012 level):")
display(pv.round(0).T)

**Scenario output:** the predicted busiest hour (clear weather, typical temperature for the month, 2012 demand level):
- **Working days:** 445 (January) → **880 (October)**.
- **Non-working days:** 234 (January) → 699 (October).

The peak requirement **doubles between winter and autumn**. This table (12 months × 2 day types × 3 weather types × 24 hours, point and P90) is saved for the prototype, which also calls the model live for any date and weather.

*Deployment note:* the saved model is refit on all of 2011–2012 (more data, most recent level). The evaluation numbers above come from the train-only model; that is standard practice.

## 10. Save the model bundle

In [ ]:
bundle_path = "models/best_model.joblib"
mt.save_bundle(bundle_path, model=deploy_model, features=F, model_name=final_name,
               planning_multipliers=mult, climate=climate, trained_on="2011-01-01..2012-12-31",
               test_metrics=test_table.loc[final_name].to_dict(), cv_metrics=cv_summary.loc[final_name].to_dict(),
               notes="Point forecast of hourly rentals; demand level of 2012 (yr capped at 1).")
b = mt.load_bundle(bundle_path)
chk = mt.scenario_frame("2012-09-12", hours=[8, 17], weather=1, climate=b["climate"])
print("Reloaded OK -> Wed 12 Sep 2012, clear: predicted rentals at 08:00 and 17:00 =",
      np.round(b["model"].predict(chk[b["features"]])).tolist(), "| sklearn", b["sklearn_version"])
print("Saved:", bundle_path, f"({os.path.getsize(bundle_path) / 1e6:.1f} MB)")

## 11. Cost of under- vs over-prediction

| Error | What happens | Cost |
|---|---|---|
| **Under-predict** a peak | Too few bikes at busy stations → riders find none | Lost rides and revenue, complaints, **churn of registered members** (the commuters), brand damage |
| **Over-predict** | Bikes positioned that aren't used | Extra truck time, some idle bikes – cheap and recoverable the next day |

Because under-prediction is costlier, the model was chosen on **peak error and bias**, not on R² alone. Plans use the **P90 forecast at peaks** and the point forecast elsewhere. The 81% under-prediction rate at test peaks is the main weakness; monthly retraining and the P90 buffer are the mitigations.

## 12. Recommendation to the CEO (based on this evidence)

**Don't buy more bikes yet. First manage *when* and *where* bikes are, then decide.**
1. **The evidence points to timing and placement.** On working days, 58% of rentals happen in 7 hours, and the evening peak has ~2.6× the bikes on the road that midday has (241 vs 92). Idle bikes at noon are built into that pattern. Buying bikes adds capacity at noon and at night too, when it is not needed.
2. **Demand is predictable.** The forecast error is ~17% of volume (R² 0.91 on unseen months). With a P90 buffer it covers ~90% of hours. That is good enough to plan rebalancing:
   - Each evening, forecast the next day.
   - Pre-position bikes before 07:00 and before 16:00.
   - Do the moves at night (only 3% of rentals) and in the midday trough.
   - Scale the plan with the weather forecast (rain −45%) and the season (2× swing).
3. **Prove it with the switchback pilot** (Phase 2, §4.6): 4–6 weeks, measuring empty-station minutes at peaks.
4. **Buy only if the evidence says so:**
   - stations stay empty at peaks even after rebalancing, or
   - P90 bikes-on-road at the peak approaches the usable fleet (fleet minus bikes in repair).

   If either happens, buy **targeted** bikes for those stations, and consider **seasonal leasing**, given the 2× seasonal swing and +64% growth.
5. **Data to collect:** station-level dock status (empty/full minutes), trip records (origin, destination, duration), failed rental attempts, fleet size and repair status, rebalancing logs.

**What this analysis cannot claim:** it cannot prove shortages exist, or where they occur. The data has only completed rentals, city-wide.

## Phase 3 summary

**Completed**
- 12 candidate models compared with time-ordered CV. **Gradient boosting (depth 6)** was chosen on MAE, peak error and bias, not R² alone.
- Test (Jul–Dec 2012): **MAE 43.1, R² 0.91, WAPE 17%**, a 60% lower error than the business baseline. Weakness: under-prediction at peaks (bias −50 at rush hours).
- Lag features tested and rejected. Random split shown to be over-optimistic (MAE 23). Monthly retraining shown to help (MAE 39, half the bias).
- P90 planning forecast via calibrated multipliers: **89% coverage** (96% at rush hours). Quantile GB tried and rejected (65%).
- Little's-law translation to bikes on the road, with labelled assumptions, plus a fleet decision test.
- Saved: `models/best_model.joblib` (model, features, multipliers, climate defaults, metrics); scenario table; 9 result CSVs; charts 15–20.

**Next – Phase 4:** the natural-language operations assistant, built on this saved model.

**Limitations**
- Systematic peak under-prediction under growth.
- No storm/event information.
- Trip time is assumed.
- Fleet size can't be computed without station data.
- The model's demand level is capped at 2012.